<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [1]:
from pathlib import Path
from tempfile import gettempdir

from sklearn.datasets import fetch_openml, load_digits
from sklearn.model_selection import train_test_split


def load_data(seed=42):
    """Carrega MNIST quando possivel e usa fallback local se necessario."""
    cache_dir = Path(gettempdir()) / "ml_openml_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    openml_cache = cache_dir / "openml"

    try:
        if not openml_cache.exists():
            raise FileNotFoundError("OpenML cache not available locally.")
        X, y = fetch_openml(
            "mnist_784",
            version=1,
            as_frame=False,
            return_X_y=True,
            data_home=str(cache_dir),
        )
        y = y.astype(int)
    except Exception:
        digits = load_digits()
        X, y = digits.data, digits.target

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y,
    )

    return X_train, X_test, y_train, y_test

In [2]:
X_train, X_test, y_train, y_test = load_data(seed=42)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (56000, 784), X_test: (14000, 784)
y_train: (56000,), y_test: (14000,)


## Resposta (normalização)

Para modelos baseados em árvore de decisão (Random Forest, AdaBoost com base em DecisionTree), a normalização não é estritamente necessária porque os splits são invariante a escala. No entanto, é boa prática em um pipeline geral (e obrigatória para modelos de distância/gradiente) para garantir consistência e facilitar comparações com outros algoritmos.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [3]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

In [4]:
rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)
print(rf_model)
print(ab_model)

RandomForestClassifier(random_state=42)
AdaBoostClassifier(random_state=42)


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [5]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

In [6]:
rf_acc = evaluate(rf_model, X_test, y_test)
ab_acc = evaluate(ab_model, X_test, y_test)
print(f"Random Forest accuracy: {rf_acc:.4f}")
print(f"AdaBoost accuracy: {ab_acc:.4f}")

Random Forest accuracy: 0.9672
AdaBoost accuracy: 0.6681


# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [7]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    return evaluate(model, X_test, y_test)

In [8]:
rf_pipeline_acc = run_pipeline(model_type="rf", seed=42)
ab_pipeline_acc = run_pipeline(model_type="ab", seed=42)
print(f"Pipeline Random Forest accuracy: {rf_pipeline_acc:.4f}")
print(f"Pipeline AdaBoost accuracy: {ab_pipeline_acc:.4f}")

Pipeline Random Forest accuracy: 0.9672
Pipeline AdaBoost accuracy: 0.6681


**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

O overfitting começa quando aumentar a profundidade melhora o treino, mas piora o teste. Com `max_depth=None`, a árvore cresce sem limite e pode memorizar os dados de treino, por isso consegue chegar a 100%.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

rf_pred = rf_model.predict(X_test)
ab_pred = ab_model.predict(X_test)

rf_metrics = {
    "accuracy": accuracy_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred, average="weighted"),
    "recall": recall_score(y_test, rf_pred, average="weighted"),
    "f1": f1_score(y_test, rf_pred, average="weighted"),
}

ab_metrics = {
    "accuracy": accuracy_score(y_test, ab_pred),
    "precision": precision_score(y_test, ab_pred, average="weighted"),
    "recall": recall_score(y_test, ab_pred, average="weighted"),
    "f1": f1_score(y_test, ab_pred, average="weighted"),
}

print("Random Forest:")
for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nAdaBoost:")
for metric, value in ab_metrics.items():
    print(f"{metric}: {value:.4f}")

Random Forest:
accuracy: 0.9672
precision: 0.9672
recall: 0.9672
f1: 0.9672

AdaBoost:
accuracy: 0.6681
precision: 0.6884
recall: 0.6681
f1: 0.6711


O Random Forest apresentou melhor desempenho inicial, com métricas superiores às do AdaBoost.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
rf_seed_42 = run_pipeline(model_type="rf", seed=42)
rf_seed_7 = run_pipeline(model_type="rf", seed=7)
ab_seed_42 = run_pipeline(model_type="ab", seed=42)
ab_seed_7 = run_pipeline(model_type="ab", seed=7)

print(f"Random Forest | seed=42: {rf_seed_42:.4f}")
print(f"Random Forest | seed=7: {rf_seed_7:.4f}")
print(f"AdaBoost      | seed=42: {ab_seed_42:.4f}")
print(f"AdaBoost      | seed=7: {ab_seed_7:.4f}")

Os resultados podem mudar com seeds diferentes, porque há aleatoriedade na divisão dos dados e no treinamento. Mesmo assim, o experimento é reprodutível, pois a mesma seed sempre reproduz o mesmo resultado.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)

train_acc = rf_model.score(X_train, y_train)
test_acc = rf_model.score(X_test, y_test)

print(f"Random Forest train accuracy: {train_acc:.4f}")
print(f"Random Forest test accuracy: {test_acc:.4f}")

Sim, existe overfitting quando a acurácia de treino fica maior que a de teste. O Random Forest tende a sofrer menos com isso do que uma árvore isolada.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_estimators = [10, 50, 100]
ab_estimators = [10, 50, 100]

print("Random Forest:")
for n in rf_estimators:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"n_estimators={n}: {acc:.4f}")

print("\nAdaBoost:")
for n in ab_estimators:
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"n_estimators={n}: {acc:.4f}")

O desempenho muda, mas não de forma muito drástica. O AdaBoost tende a ser mais sensível a mudanças em `n_estimators` do que o Random Forest.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. Não. A acurácia ajuda, mas pode esconder erros em classes específicas. Por isso, também é importante analisar precisão, recall e F1-score

2. Fixando seeds e repetindo o experimento com valores diferentes. Se os resultados continuarem parecidos, há mais evidência de estabilidade

3. Usar apenas uma divisão treino/teste pode gerar uma avaliação instável. Além disso, testar poucos hiperparâmetros limita a comparação entre os modelos

4. Ele é razoavelmente confiável, porque segue um fluxo reprodutível e organizado. Ainda assim, pode ser melhorado com validação cruzada e ajuste mais cuidadoso de hiperparâmetros